<a href="https://colab.research.google.com/github/get2rayan/agentic-rag/blob/main/5_faiss_ragas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install faiss-gpu
!pip install openai
!pip install datasets
!pip install ragas

In [8]:
from google.colab import userdata
import os
import faiss
from openai import OpenAI
import numpy as np
import json
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [9]:
# load json dataset
with open('sample_data/recipes_dataset.json', 'r') as f:
    data = json.loads(f.read().lower())

print(len(data))
print(data[0])

55
{'id': 1, 'name': 'classic margherita pizza', 'cuisine': 'italian', 'ingredients': ['2 cups all-purpose flour', '1 teaspoon instant yeast', '1 teaspoon salt', '3/4 cup warm water', '1 tablespoon olive oil', '1/2 cup tomato sauce', '1 cup fresh mozzarella cheese', 'fresh basil leaves', 'olive oil for drizzling'], 'instructions': 'mix flour, yeast, salt, and water to form a dough. knead for 10 minutes, then let rise for 1 hour. roll out dough and spread tomato sauce. add mozzarella cheese and fresh basil. bake at 475°f (245°c) for 12-15 minutes. drizzle with olive oil before serving.', 'cooktime_minutes': 65, 'nutrition': {'calories': 320, 'protein': 12, 'carbohydrates': 45, 'fat': 14, 'fiber': 3, 'sodium': 620}}


In [10]:
model = "text-embedding-3-small"
openai = OpenAI()

In [11]:
def get_metadata(recipe):
    return {
        # "id": recipe['id'],
        "name": recipe['name'],
        "cooktime_minutes": recipe['cooktime_minutes'],
        "cuisine": recipe['cuisine'],
        "nutrition": json.dumps(recipe['nutrition']),
        "ingredients": ", ".join(recipe['ingredients']),
        # "instructions": recipe['instructions']
    }


def get_embedding(text):
    embedding = openai.embeddings.create(input=text, model=model).data[0].embedding
    return embedding


embeddings = []
ids=[]
documents = []
metadata = []

for recipe in data:
    ids.append(f"id_{recipe['id']}")
    text_to_encode = f"recipe name: {recipe['name']}; cooktime in minutes: {recipe['cooktime_minutes']}; cuisine: {recipe['cuisine']}; nutrition info: {', '.join(f'{k}- {v}' for k, v in recipe['nutrition'].items())}; ingredients: {', '.join(recipe['ingredients'])}; instructions: {recipe['instructions']}"
    documents.append(text_to_encode)
    # embedding = model.encode(text_to_encode).tolist()
    embedding = get_embedding(text_to_encode)
    embeddings.append(embedding)
    metadata.append(get_metadata(recipe))

In [12]:
print(len(embeddings), len(embeddings[0]), len(embeddings[1]), len(embeddings[2]), len(embeddings[3]), len(embeddings[4]))

55 1536 1536 1536 1536 1536


In [13]:
# Before adding to FAISS index, we need to convert the embeddings to a numpy array of type float32
embeddings = np.array(embeddings).astype('float32')
print(embeddings.shape, embeddings.dtype, embeddings.shape[1])

index = faiss.IndexFlatIP(embeddings.shape[1])
faiss.normalize_L2(embeddings)
index.add(embeddings)


(55, 1536) float32 1536


In [14]:
# docs = [
#     "Paris is the capital and most populous city of France. The city is famed for the Eiffel Tower.",
#     "Jane Austen was an English novelist best known for 'Pride and Prejudice' and 'Sense and Sensibility'.",
#     "The Great Wall of China is a series of fortifications built to protect the ancient Chinese states.",
#     "Mount Everest, part of the Himalayas, is Earth’s highest mountain above sea level.",
#     "Mike loves the color pink more than any other color."
# ]

# client = OpenAI()

# def get_embedding(text):
#     response = client.embeddings.create(model="text-embedding-3-small", input=text)
#     return response.data[0].embedding

# embeddings = np.array([get_embedding(d) for d in docs]).astype('float32')
# print(embeddings.shape, embeddings.dtype, embeddings.shape[1])
# index = faiss.IndexFlatIP(embeddings.shape[1])
# faiss.normalize_L2(embeddings)
# index.add(embeddings)

In [15]:
query = "i need a sweet recipe with least calories"

def retrieve(query, k=3):
    query_embedding = get_embedding(query)
    query_embedding = np.array([query_embedding]).astype('float32')
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, k)
    print(f"index shape: {indices.shape}, distances shape: {distances.shape}")
    print(indices, distances)
    print("---------------------------------------------------------------")
    results = []
    for i in range(k):
        idx = indices[0][i]
        print(f"Index: {idx}, Distance: {distances[0][i]}")
        if idx < len(documents):
            results.append({
                "document": documents[idx],
                "metadata": metadata[idx],
                "distance": distances[0][i]
            })
    print("\n------------------------------------\n")
    return results


results = retrieve(query, k=5)

for result in results:
    print(f"Document: {result['document']}")
    print(f"Metadata: {result['metadata']}")
    print(f"Distance: {result['distance']}")
    print("-----")

index shape: (1, 5), distances shape: (1, 5)
[[54 23 46 49  4]] [[0.42846653 0.412803   0.3956722  0.3885268  0.3869907 ]]
---------------------------------------------------------------
Index: 54, Distance: 0.4284665286540985
Index: 23, Distance: 0.4128029942512512
Index: 46, Distance: 0.3956722021102905
Index: 49, Distance: 0.3885267972946167
Index: 4, Distance: 0.3869906961917877

------------------------------------

Document: recipe name: apple crisp; cooktime in minutes: 50; cuisine: american; nutrition info: calories- 320, protein- 3, carbohydrates- 56, fat- 12, fiber- 5, sodium- 80; ingredients: 6 apples, peeled and sliced, 1/2 cup sugar, 1 teaspoon cinnamon, 1/4 teaspoon nutmeg, 1 cup rolled oats, 1/2 cup flour, 1/2 cup brown sugar, 1/2 cup cold butter, cubed, vanilla ice cream for serving; instructions: toss apples with sugar, cinnamon, and nutmeg. place in a baking dish. mix oats, flour, and brown sugar. cut in cold butter until crumbly. sprinkle topping over apples. bake at

In [16]:
context = "\n".join([f" - {result['document']}" for result in results])

def get_instructions(query, context):
    instructions = f"""You are a helpful assistant who answers user's queries based **only** with the facts provided in the context.

    context:
    {context}

    Question:
    {query}

    Answer:
    """
    return instructions


print(get_instructions(query, context))

You are a helpful assistant who answers user's queries based **only** with the facts provided in the context.

    context:
     - recipe name: apple crisp; cooktime in minutes: 50; cuisine: american; nutrition info: calories- 320, protein- 3, carbohydrates- 56, fat- 12, fiber- 5, sodium- 80; ingredients: 6 apples, peeled and sliced, 1/2 cup sugar, 1 teaspoon cinnamon, 1/4 teaspoon nutmeg, 1 cup rolled oats, 1/2 cup flour, 1/2 cup brown sugar, 1/2 cup cold butter, cubed, vanilla ice cream for serving; instructions: toss apples with sugar, cinnamon, and nutmeg. place in a baking dish. mix oats, flour, and brown sugar. cut in cold butter until crumbly. sprinkle topping over apples. bake at 375°f for 35-40 minutes until bubbly and golden. serve warm with vanilla ice cream.
 - recipe name: banana pancakes; cooktime in minutes: 20; cuisine: american; nutrition info: calories- 280, protein- 8, carbohydrates- 48, fat- 7, fiber- 3, sodium- 320; ingredients: 1 ripe banana, mashed, 1 cup all-pur

In [17]:
chatModel = "gpt-4o-mini"

def generate_answer(instructions):
    aiResponse = openai.chat.completions.create(model=chatModel, messages=[{'role': 'system', 'content': instructions}])
    return aiResponse.choices[0].message.content.strip()


## Evaluation

In [18]:
from datasets import Dataset

questions = [
    "I want an indian recipe made of rice and chicken",
    "Give me recipe for a dessert which has least calories",
    "What is a non-american sweet dish that I can make in a short time",
    "Provide a south american seafood dish recipe ",
    "Give me a recipe for an indian vegetarian dish"
]

ground_truths=[
    "Biryani is an indian cuisine that you can make with rice and chicken",
    "Chocolate Chip cookies is one recipe that has less calories",
    "Baklava is a middle eastern dessert that you can make in a short time",
    "Ceviche is a peruvian dish that is made with white fish",
    "Palak Paneer or Chickpea Cury are some of indian vegetarian dish"
]

rows=[]

for question, ground_truth in zip(questions, ground_truths):
    retrieval_results=retrieve(question, k=2)
    context = "\n".join([f" - {result['document']}" for result in retrieval_results])
    instruction = get_instructions(question, context)
    answer = generate_answer(instruction)
    rows.append({
        'question': question,
        'context': context,
        'answer': answer,
        'reference': ground_truth
    })

evaluation_dataset=Dataset.from_list(rows)

print(rows)

index shape: (1, 2), distances shape: (1, 2)
[[50 14]] [[0.5765233 0.5140276]]
---------------------------------------------------------------
Index: 50, Distance: 0.5765233039855957
Index: 14, Distance: 0.5140275955200195

------------------------------------

index shape: (1, 2), distances shape: (1, 2)
[[54 49]] [[0.44242394 0.4371143 ]]
---------------------------------------------------------------
Index: 54, Distance: 0.442423939704895
Index: 49, Distance: 0.43711429834365845

------------------------------------

index shape: (1, 2), distances shape: (1, 2)
[[54 49]] [[0.4394315  0.42557597]]
---------------------------------------------------------------
Index: 54, Distance: 0.43943148851394653
Index: 49, Distance: 0.42557597160339355

------------------------------------

index shape: (1, 2), distances shape: (1, 2)
[[40 10]] [[0.5498396  0.48585737]]
---------------------------------------------------------------
Index: 40, Distance: 0.5498396158218384
Index: 10, Distance: 0.

In [21]:
!pip install --upgrade langchain langchain-community langchain-google-vertexai ragas

import sys
import os
import numpy as np
from datasets import Dataset
from langchain_google_vertexai import ChatVertexAI, VertexAI

# Mock the legacy import path that ragas 0.4.3 expects
try:
    import langchain_community.chat_models as chat_models
    sys.modules["langchain_community.chat_models.vertexai"] = sys.modules["langchain_google_vertexai"]

    import langchain_community.llms as llms
    sys.modules["langchain_community.llms.vertexai"] = sys.modules["langchain_google_vertexai"]
except Exception:
    pass

from ragas import evaluate
from ragas.metrics import (
    answer_correctness,
    answer_relevancy,
    faithfulness,
    context_precision,
    context_recall,
)

rows=[]
for question, ground_truth in zip(questions, ground_truths):
    retrieval_results = retrieve(question, k=2)
    context_text = "\n".join([f" - {result['document']}" for result in retrieval_results])
    instruction = get_instructions(question, context_text)
    answer = generate_answer(instruction)
    rows.append({
        'user_input': question,
        'retrieved_contexts': [result['document'] for result in retrieval_results],
        'response': answer,
        'reference': ground_truth
    })

evaluation_dataset = Dataset.from_list(rows)

# Execute evaluation
scores = evaluate(
    evaluation_dataset,
    metrics=[
        answer_correctness,
        answer_relevancy,
        context_precision
    ],
)

print(scores)

/tmp/ipykernel_4979/2279744732.py:20: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
/tmp/ipykernel_4979/2279744732.py:20: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_4979/2279744732.py:20: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_4979/2279744732.py:20: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will b

index shape: (1, 2), distances shape: (1, 2)
[[50 14]] [[0.5765233 0.5140276]]
---------------------------------------------------------------
Index: 50, Distance: 0.5765233039855957
Index: 14, Distance: 0.5140275955200195

------------------------------------

index shape: (1, 2), distances shape: (1, 2)
[[54 49]] [[0.44242394 0.4371143 ]]
---------------------------------------------------------------
Index: 54, Distance: 0.442423939704895
Index: 49, Distance: 0.43711429834365845

------------------------------------

index shape: (1, 2), distances shape: (1, 2)
[[54 49]] [[0.4394315  0.42557597]]
---------------------------------------------------------------
Index: 54, Distance: 0.43943148851394653
Index: 49, Distance: 0.42557597160339355

------------------------------------

index shape: (1, 2), distances shape: (1, 2)
[[40 10]] [[0.5498396  0.48585737]]
---------------------------------------------------------------
Index: 40, Distance: 0.5498396158218384
Index: 10, Distance: 0.

Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: TypeError(Cannot use aembed_text() with a synchronous client. Use embed_text() instead.)
ERROR:ragas.executor:Exception raised in Job[1]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[3]: TypeError(Cannot use aembed_text() with a synchronous client. Use embed_text() instead.)
ERROR:ragas.executor:Exception raised in Job[4]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[6]: TypeError(Cannot use aembed_text() with a synchronous client. Use embed_text() instead.)
ERROR:ragas.executor:Exception raised in Job[7]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[9]: IncompleteOutputException(The output is incomplete due to a max_tokens length limit.)
ERROR:ragas.executor:Exception raised in Job[10]: AttributeError('OpenAIEmbeddings' o

{'answer_correctness': nan, 'answer_relevancy': nan, 'context_precision': 0.6000}
